# 02 - Data Quality Assessment

## Tujuan

Notebook ini memeriksa kualitas data dan pola duplicate sebagai dasar entity resolution.
Dataset mentah tidak diubah dan tidak ditimpa.

## Output utama

- Profil struktur, tipe data, missing value, dan cardinality
- Validasi identifier dan exact duplicate
- Pemeriksaan format email, telepon, nama, alamat, dan tanggal
- Analisis pola record berdasarkan `customer_id`
- Kandidat field untuk tahap matching

In [23]:
from pathlib import Path
import re
import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'raw' / 'crm_50000_customers_dirty_v3.csv',
    Path.cwd().parent / 'data' / 'raw' / 'crm_50000_customers_dirty_v3.csv',
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('CSV dataset tidak ditemukan.')

df = pd.read_csv(DATA_PATH)
print(f'File: {DATA_PATH}')
print(f'Shape: {df.shape}')

File: c:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\raw\crm_50000_customers_dirty_v3.csv
Shape: (50000, 14)


In [17]:
# Struktur dan tipe data
structure_summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null': df.notna().sum().values,
    'unique': df.nunique(dropna=True).values,
})
structure_summary

,column,dtype,non_null,unique
0,customer_id,str,50000,48200
1,first_name,str,50000,5429
2,last_name,str,50000,7228
3,email,str,48960,46363
4,phone_number,str,50000,46777
5,gender,str,50000,3
6,dob,str,50000,17733
7,signup_date,str,50000,4383
8,address,str,50000,48200
9,city,str,50000,24534


In [18]:
# Missing value dan cardinality
quality_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': df.isna().mean().mul(100),
    'unique_count': df.nunique(dropna=True),
    'unique_percentage': df.nunique(dropna=True).div(len(df)).mul(100),
}).sort_values('missing_percentage', ascending=False)
quality_summary.round(2)

,missing_count,missing_percentage,unique_count,unique_percentage
email,1040,2.08,46363,92.73
customer_id,0,0.00,48200,96.40
first_name,0,0.00,5429,10.86
last_name,0,0.00,7228,14.46
phone_number,0,0.00,46777,93.55
gender,0,0.00,3,0.01
dob,0,0.00,17733,35.47
signup_date,0,0.00,4383,8.77
address,0,0.00,48200,96.40
city,0,0.00,24534,49.07


In [19]:
# Identifier dan exact duplicate
exact_duplicate_count = int(df.duplicated().sum())
duplicate_id_summary = pd.DataFrame()
if 'customer_id' in df.columns:
    duplicate_id_summary = (
        df.groupby('customer_id', dropna=False).size()
        .reset_index(name='row_count')
        .query('row_count > 1')
        .sort_values('row_count', ascending=False)
    )

print(f'Exact duplicate rows: {exact_duplicate_count:,} ({exact_duplicate_count / len(df):.2%})')
print(f'customer_id unique: {df.customer_id.nunique(dropna=True):,}')
print(f'Rows with repeated customer_id after first occurrence: {df.customer_id.duplicated().sum():,}')
print(f'customer_id groups repeated: {len(duplicate_id_summary):,}')
print(f'Maximum rows per customer_id: {duplicate_id_summary["row_count"].max():,}')

# Tampilkan hanya frekuensi; raw customer_id tidak diekspos.
duplicate_id_summary[['row_count']].head(20).reset_index(drop=True)

Exact duplicate rows: 1,021 (2.04%)
customer_id unique: 48,200
Rows with repeated customer_id after first occurrence: 1,800
customer_id groups repeated: 1,734
Maximum rows per customer_id: 4


,row_count
0,4
1,3
2,3
3,3
4,3
5,3
6,3
7,3
8,3
9,3


In [20]:
# Profiling field identitas
def profile_text_field(series: pd.Series) -> dict:
    values = series.dropna().astype(str)
    return {
        'non_null': len(values),
        'raw_unique': values.nunique(),
        'blank': int(values.str.strip().eq('').sum()),
    }

field_profiles = {}
for column in ['email', 'phone_number', 'first_name', 'last_name', 'address']:
    if column in df.columns:
        field_profiles[column] = profile_text_field(df[column])

email = df['email'].dropna().astype(str)
email_pattern = r'[^@\s]+@[^@\s]+\.[^@\s]+'
field_profiles['email'].update({
    'simple_valid_format': int(email.str.fullmatch(email_pattern).sum()),
    'simple_invalid_format': int((~email.str.fullmatch(email_pattern)).sum()),
    'casefold_unique': email.str.casefold().nunique(),
})

phone = df['phone_number'].dropna().astype(str)
phone_digits = phone.str.replace(r'\D', '', regex=True)
field_profiles['phone_number'].update({
    'digits_unique': phone_digits.nunique(),
    'digit_length_min': int(phone_digits.str.len().min()),
    'digit_length_max': int(phone_digits.str.len().max()),
})

full_name = (df['first_name'].fillna('') + ' ' + df['last_name'].fillna('')).str.strip()
normalized_name = full_name.str.lower().str.replace(r'[^a-z0-9]', '', regex=True)
field_profiles['full_name'] = {
    'raw_unique': full_name.nunique(),
    'normalized_unique': normalized_name.nunique(),
    'normalization_collision_reduction': full_name.nunique() - normalized_name.nunique(),
}

address = df['address'].astype(str)
normalized_address = address.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
field_profiles['address'].update({
    'space_normalized_unique': normalized_address.nunique(),
    'space_normalization_reduction': address.nunique() - normalized_address.nunique(),
})

pd.DataFrame(field_profiles).T

,non_null,raw_unique,blank,simple_valid_format,simple_invalid_format,casefold_unique,digits_unique,digit_length_min,digit_length_max,space_normalized_unique,space_normalization_reduction,normalized_unique,normalization_collision_reduction
email,48960.0,46363.0,0.0,48960.0,0.0,46363.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phone_number,50000.0,46777.0,0.0,NaN,NaN,NaN,46777.0,10.0,18.0,NaN,NaN,NaN,NaN
first_name,50000.0,5429.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_name,50000.0,7228.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,50000.0,48200.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,48200.0,0.0,NaN,NaN
full_name,NaN,42585.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40254.0,2331.0


In [21]:
# Profiling tanggal tanpa mengubah kolom asli
date_profiles = []
for column in ['dob', 'signup_date']:
    if column in df.columns:
        parsed = pd.to_datetime(df[column], errors='coerce')
        date_profiles.append({
            'column': column,
            'parseable': int(parsed.notna().sum()),
            'unparseable': int(parsed.isna().sum()),
            'raw_unique': int(df[column].nunique(dropna=True)),
            'parsed_unique': int(parsed.nunique(dropna=True)),
            'min': parsed.min(),
            'max': parsed.max(),
        })
pd.DataFrame(date_profiles)

,column,parseable,unparseable,raw_unique,parsed_unique,min,max
0,dob,50000,0,17733,17733,1954-12-20,2007-12-07
1,signup_date,50000,0,4383,4383,2013-12-02,2025-12-01


In [22]:
# Kandidat field entity resolution berdasarkan cardinality dan completeness
candidate_fields = pd.DataFrame({
    'field': df.columns,
    'unique_count': df.nunique(dropna=True).values,
    'unique_ratio': (df.nunique(dropna=True) / len(df)).values,
    'missing_ratio': df.isna().mean().values,
}).sort_values(['unique_ratio', 'missing_ratio'], ascending=[False, True])
candidate_fields

,field,unique_count,unique_ratio,missing_ratio
0,customer_id,48200,0.96400,0.0000
8,address,48200,0.96400,0.0000
12,device_id(s),48200,0.96400,0.0000
4,phone_number,46777,0.93554,0.0000
3,email,46363,0.92726,0.0208
9,city,24534,0.49068,0.0000
6,dob,17733,0.35466,0.0000
2,last_name,7228,0.14456,0.0000
1,first_name,5429,0.10858,0.0000
7,signup_date,4383,0.08766,0.0000


## Rekomendasi tahap berikutnya

1. Tetapkan aturan standardisasi email, telepon, nama, alamat, dan tanggal pada kolom baru.
2. Pisahkan exact duplicate dari duplicate yang hanya berbagi `customer_id` atau field tertentu.
3. Audit record dengan `customer_id` berulang untuk menentukan apakah identifier memang tidak unik.
4. Gunakan kombinasi `email`, `phone_number`, nama, `dob`, dan alamat sebagai kandidat blocking/matching, bukan satu field secara membabi buta.
5. Cari atau buat ground truth sebelum mengukur precision, recall, atau threshold fuzzy matching.
6. Jangan menimpa dataset mentah; simpan hasil transformasi ke dataset turunan.